# IEEE300 EMT benchmark: SSN vs aggregate vs grouped SSN

Three EMT formulations of the MATPOWER IEEE300 case (300 buses, 69 conventional VBR generators, 411 branches), all run for **10 s simulated time** with **Δt = 50 μs** (200,000 steps). The generator model and operating point are held fixed; the passive network is represented as individual SSN components, one fused aggregate, or one grouped passive SSN using shared maps. Each profile is a single run, so these are direct measurements, not multi-run averages.

The case uses the IEEE300 steady-state data with the same generic 4th-order synchronous-machine setup used in the IEEE118 EMT benchmark; generator dynamic parameters are not a separate IEEE300 dynamic-model dataset.

In [ ]:
from pathlib import Path
import lzma
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("examples/Notebooks/Benchmarks/data/IEEE300_EMT_10s_50us")
if not DATA.exists():
    DATA = Path("data/IEEE300_EMT_10s_50us")
CONFIGS = ["ssn", "aggregate", "grouped_ssn"]
LABELS = {
    "ssn": "SSN components",
    "aggregate": "Aggregate SSN",
    "grouped_ssn": "Grouped passive SSN (shared maps)",
}
COLORS = {"ssn": "#F58518", "aggregate": "#59A14F", "grouped_ssn": "#B279A2"}
EXPECTED_STEPS = 200_000

wall = pd.read_csv(DATA / "wall_times.csv").set_index("configuration")["wall_s"]
profiles = {}
for config in CONFIGS:
    tasks = pd.read_csv(DATA / f"{config}_TaskTimes.csv")
    tasks["stage"] = tasks["task"].str.rsplit(".", n=1).str[-1]
    tasks["component"] = tasks["task"].str.rsplit(".", n=1).str[0]
    tasks["total_s"] = tasks["total_ns"] / 1e9
    tasks["avg_us"] = tasks["avg_ns"] / 1000.0
    with lzma.open(DATA / f"{config}_StepTimes.log.xz", "rt") as step_file:
        steps_ms = np.loadtxt(step_file, skiprows=1) * 1000.0
    phase = pd.read_csv(DATA / f"{config}_SolverPhaseTimes.csv")
    matrix = pd.read_csv(DATA / f"{config}_SolverMatrixStats.csv")
    profiles[config] = {
        "tasks": tasks,
        "steps_ms": steps_ms,
        "phase": phase,
        "matrix": matrix,
    }

stage_rows = []
for config in CONFIGS:
    tasks = profiles[config]["tasks"]
    row = {"configuration": LABELS[config], "wall_s": wall[config]}
    for stage, key in [
        ("MnaPreStep", "pre_step_s"),
        ("MnaPostStep", "post_step_s"),
        ("Solve", "solve_s"),
    ]:
        selected = tasks[tasks["stage"] == stage]
        row[key] = selected["total_s"].sum()
        row[key.replace("_s", "_calls")] = int(selected["calls"].sum())
    row["steps"] = len(profiles[config]["steps_ms"])
    stage_rows.append(row)
summary = pd.DataFrame(stage_rows).set_index("configuration")
summary.round(3)

In [ ]:
# Main comparison: profiled pre-step, post-step, Solve task and whole-process wall time.
ax = summary[["pre_step_s", "post_step_s", "solve_s", "wall_s"]].plot.bar(
    figsize=(12, 5.5), color=["#70AD47", "#FFC000", "#A5A5A5", "#4472C4"], width=0.78
)
ax.set_title("IEEE300 EMT timing | Δt = 50 μs | T_sim = 10 s")
ax.set_ylabel("seconds")
ax.set_xlabel("network representation (one profiled run each)")
ax.grid(axis="y", alpha=0.25)
ax.legend(["Pre-step", "Post-step", "Solve", "Wall-to-wall"], ncol=4, loc="upper left")
ax.set_xticklabels(ax.get_xticklabels(), rotation=12, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=2, fontsize=8)
plt.tight_layout()
plt.show()
display(summary.round(3))

In [ ]:
# Component-family timing: compare like work by stage, rather than individual devices.
def family(component):
    if component.startswith("Gen"):
        return "Generators"
    if component.startswith("Trafo"):
        return "Transformers"
    if component.startswith("LinearNetworkGroup"):
        return "Grouped passive network"
    if component.startswith("LinearNetwork"):
        return "Aggregate passive network"
    if component.startswith("Line"):
        return "Pi-lines"
    if component.startswith(("ImpedanceLoad", "Load", "Shunt")):
        return "Loads + shunts"
    return "Other"


family_rows = []
for config in CONFIGS:
    frame = profiles[config]["tasks"].copy()
    frame = frame[frame["stage"].isin(["MnaPreStep", "MnaPostStep"])]
    frame["family"] = frame["component"].map(family)
    family_rows.append(
        frame.groupby(["family", "stage"])["total_s"]
        .sum()
        .rename_axis(index={0: "family", 1: "stage"})
        .reset_index()
        .assign(configuration=config)
    )
family_totals = pd.concat(family_rows, ignore_index=True)
order = [
    "Pi-lines",
    "Generators",
    "Transformers",
    "Loads + shunts",
    "Aggregate passive network",
    "Grouped passive network",
]
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for ax, stage, title in zip(
    axes,
    ["MnaPreStep", "MnaPostStep"],
    ["Pre-step by component family", "Post-step by component family"],
):
    p = (
        family_totals[family_totals.stage == stage]
        .pivot(index="family", columns="configuration", values="total_s")
        .reindex(order)
        .fillna(0)
    )
    p = p.rename(columns=LABELS).reindex(columns=[LABELS[c] for c in CONFIGS])
    p.plot.bar(ax=ax, color=[COLORS[c] for c in CONFIGS], width=0.8)
    ax.set_title(title)
    ax.set_ylabel("cumulative seconds")
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8)
fig.suptitle("IEEE300 EMT component-family timing | Δt = 50 μs | T_sim = 10 s")
fig.tight_layout()
plt.show()

In [ ]:
# Internal solver counters. These are instrumentation subphases and can overlap;
# do not add them to each other or to the top-level task totals.
phase_names = [
    "rhs_assembly",
    "numeric_solve",
    "partial_refactorization",
    "group_history_state_multiply",
    "group_state_update",
    "dense_lu_fallback_factorization",
]
phase_rows = []
for config in CONFIGS:
    frame = profiles[config]["phase"].set_index("phase")
    for name in phase_names:
        phase_rows.append(
            {
                "configuration": LABELS[config],
                "phase": name,
                "total_s": (
                    float(frame.loc[name, "total_s"]) if name in frame.index else 0.0
                ),
                "calls": int(frame.loc[name, "calls"]) if name in frame.index else 0,
            }
        )
phase_summary = pd.DataFrame(phase_rows)
plot = phase_summary.pivot(
    index="phase", columns="configuration", values="total_s"
).reindex(phase_names)
plot = plot.rename(columns=LABELS).reindex(columns=[LABELS[c] for c in CONFIGS])
ax = plot.plot.bar(figsize=(14, 6), color=[COLORS[c] for c in CONFIGS], width=0.82)
ax.set_title(
    "Internal solver timing counters (may overlap) | IEEE300 EMT | 50 μs | 10 s"
)
ax.set_ylabel("cumulative seconds")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
display(phase_summary.sort_values(["phase", "configuration"]))

In [ ]:
# Per-step duration distribution, plus the actual reduced/global matrix sizes.
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.boxplot(
    [profiles[c]["steps_ms"] for c in CONFIGS],
    tick_labels=[LABELS[c] for c in CONFIGS],
    showfliers=False,
    showmeans=True,
)
ax.set_title("Step-time distribution | IEEE300 EMT | Δt = 50 μs | T_sim = 10 s")
ax.set_ylabel("milliseconds per step")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=12, ha="right")
plt.tight_layout()
plt.show()

matrix_rows = []
for config in CONFIGS:
    m = profiles[config]["matrix"].iloc[0]
    matrix_rows.append(
        {
            "configuration": LABELS[config],
            "rows": int(m["rows"]),
            "cols": int(m["cols"]),
            "nonzeros": int(m["nonzeros"]),
            "density": float(m["structural_density"]),
            "variable_entries": int(m["variable_entries"]),
        }
    )
display(pd.DataFrame(matrix_rows).set_index("configuration"))

In [ ]:
# Reproducibility checks and highest-cost individual scheduler tasks (with calls).
assert all(profiles[c]["steps_ms"].size == EXPECTED_STEPS for c in CONFIGS)
assert all(
    int(summary.loc[LABELS[c], "solve_calls"]) == EXPECTED_STEPS for c in CONFIGS
)
print(
    "Each configuration completed 200,000 steps (10 s at 50 μs); all three processes exited successfully."
)
for config in CONFIGS:
    print("\n", LABELS[config])
    display(
        profiles[config]["tasks"]
        .sort_values("total_s", ascending=False)[["task", "calls", "avg_us", "total_s"]]
        .head(12)
        .round(4)
    )